### ЗАДАЧА: Распределение доставок между курьерами

Логистическая команда получает пакет заявок на доставки, которые нужно распределить между курьерами на электровелосипедах.
Нужно собрать систему, которая:
- принимает корректные доставки,
- отбрасывает неправильные или небезопасные заявки,
- уменьшает доступный заряд после успешно назначенного маршрута,
- ведёт отдельный журнал ошибок,
- помогает понять, какой курьер был загружен дольше всех и какому клиенту доставили самый большой вес.

In [ ]:

from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Any


couriers = {
    'CR-1': {'zone': 'north', 'charge_min': 40, 'max_weight': 3.0},
    'CR-2': {'zone': 'south', 'charge_min': 30, 'max_weight': 2.0},
    'CR-3': {'zone': 'north', 'charge_min': 55, 'max_weight': 5.0},
}

rows = [
    'DL-100|CR-1|Clinic|1.5|12',
    'DL-101|CR-2|Cafe|2.5|10',
    'DL-102|CR-9|Lab|1.0|8',
    'DL-103|CR-1|Shop|0|6',
    'DL-104|CR-3|Village|3.5|60',
    'DL-100|CR-3|Clinic|1.0|10',
    'DL-105|CR-3|School|2.0|20',
    'DL-106|CR-2|Pharmacy|1.0|15',
]


class DeliveryError(Exception):
    pass


class RowFormatError(DeliveryError):
    pass


class CourierNotFoundError(DeliveryError):
    pass


class WeightError(DeliveryError):
    pass


class RouteTimeError(DeliveryError):
    pass


class WeightLimitError(DeliveryError):
    pass


class ChargeReserveError(DeliveryError):
    pass


class DuplicateDeliveryError(DeliveryError):
    pass


@dataclass(order=True)
class Delivery:
    route_min: int
    delivery_id: str
    courier_id: str
    client: str
    weight_kg: float


class Courier:
    def __init__(self, courier_id, zone, charge_min, max_weight):
        self.courier_id = courier_id
        self.zone = zone
        self.charge_min = charge_min
        self.max_weight = max_weight
        self.deliveries: List[Delivery] = []

    def charge_left(self):
        total = self.charge_min - self.total_route_time()
        return total

    def total_route_time(self):
        return sum(d.route_min for d in self.deliveries)

    def total_weight(self):
        return sum(d.weight_kg for d in self.deliveries)

    def assign(self, delivery: Delivery):
        if delivery.weight_kg > self.max_weight:
            raise WeightLimitError(f"Delivery weight {delivery.weight_kg} exceeds max {self.max_weight}")
        charge_after = self.charge_left() - delivery.route_min
        if charge_after < 5:
            raise ChargeReserveError(f"Not enough charge left after assignment: {charge_after} minutes")
        self.deliveries.append(delivery)
        self.deliveries.sort()


class CourierDispatchService:
    def __init__(self, couriers_dict):
        # Создаем курьеров по id
        self.couriers: Dict[str, Courier] = {}
        for cid, info in couriers_dict.items():
            self.couriers[cid] = Courier(cid, info['zone'], info['charge_min'], info['max_weight'])
        self.accepted: List[Delivery] = []
        self.errors: List[Tuple[str, str, str]] = []  # row, error_type, message
        self.processed_ids: set = set()

    def parse_delivery(self, row: str) -> Delivery:
        parts = row.split('|')
        if len(parts) != 5:
            raise RowFormatError(f"Incorrect number of parts: {len(parts)}")
        delivery_id, courier_id, client, weight_raw, route_raw = parts
        if courier_id not in self.couriers:
            raise CourierNotFoundError(f"Courier {courier_id} not found")
        try:
            weight_kg = float(weight_raw)
        except Exception as e:
            raise WeightError(f"Invalid weight: {weight_raw}") from e
        try:
            route_min = int(route_raw)
        except Exception as e:
            raise RouteTimeError(f"Invalid route time: {route_raw}") from e
        if weight_kg <= 0:
            raise WeightError("Weight must be positive")
        if route_min <= 0:
            raise RouteTimeError("Route time must be positive")
        return Delivery(route_min=route_min, delivery_id=delivery_id, courier_id=courier_id, client=client, weight_kg=weight_kg)

    def submit(self, row: str):
        try:
            delivery = self.parse_delivery(row)
            if delivery.delivery_id in self.processed_ids:
                raise DuplicateDeliveryError(f"Duplicate delivery ID {delivery.delivery_id}")
            courier = self.couriers[delivery.courier_id]
            courier.assign(delivery)
            self.processed_ids.add(delivery.delivery_id)
            self.accepted.append(delivery)
        except DeliveryError as e:
            self.errors.append((row, e.__class__.__name__, str(e)))

    def load(self, rows: List[str]):
        for row in rows:
            self.submit(row)

    def client_weights(self) -> Dict[str, float]:
        result: Dict[str, float] = {}
        for delivery in self.accepted:
            result[delivery.client] = result.get(delivery.client, 0) + delivery.weight_kg
        return result

    def top_client(self) -> Optional[Tuple[str, float]]:
        weights = self.client_weights()
        if not weights:
            return None
        max_client = max(weights.items(), key=lambda item: item[1])
        return max_client

    def busiest_courier(self) -> Optional[Tuple[str, int]]:
        if not self.couriers:
            return None
        max_courier = max(self.couriers.values(), key=lambda c: c.total_route_time())
        return (max_courier.courier_id, max_courier.total_route_time())

    def low_charge_couriers(self, threshold=15) -> List[Tuple[str, int]]:
        result = []
        for cid, c in self.couriers.items():
            lc = c.charge_left()
            if lc <= threshold:
                result.append((cid, lc))
        return result

    def find_delivery(self, delivery_id: str) -> Optional[Delivery]:
        for courier in self.couriers.values():
            for d in courier.deliveries:
                if d.delivery_id == delivery_id:
                    return d
        return None


# Инициализация сервиса и загрузка данных
service = CourierDispatchService(couriers)

# Загрузка данных
service.load(rows)

# Вывод принятых доставок
print("Accepted deliveries:")
for d in service.accepted:
    print(d)

# Вывод ошибок
print("\nErrors:")
for err in service.errors:
    print(err)

# Статистика по каждому курьеру
for cid, courier in service.couriers.items():
    print(f"\nCourier {cid}:")
    print(f"  Deliveries: {len(courier.deliveries)}")
    print(f"  Total route time: {courier.total_route_time()} min")
    print(f"  Charge left: {courier.charge_left()} min")
    print(f"  Total weight: {courier.total_weight()} kg")

# Топ клиент
top_client = service.top_client()
if top_client:
    print(f"\nTop client: {top_client[0]} with total weight {top_client[1]} kg")
else:
    print("\nNo deliveries for top client.")

# Курс с максимальным временем маршрута
busiest = service.busiest_courier()
if busiest:
    print(f"\nBusiest courier: {busiest[0]} with {busiest[1]} min total route time")
else:
    print("\nNo courier data.")

# Курьеры с зарядом ниже или равным порогу
low_charge = service.low_charge_couriers()
print("\nCouriers with low charge:")
for cid, charge in low_charge:
    print(f"  {cid}: {charge} min left")

# Поиск конкретной доставки
delivery_id = 'DL-105'
found_delivery = service.find_delivery(delivery_id)
print(f"\nDelivery with ID '{delivery_id}': {found_delivery}")